In [24]:
from constants import decimated_url, full_url
from pygeodesy.sphericalTrigonometry import LatLon
import pandas as pd
import numpy as np
import math
expocode = "11BE19950303"
# expocode = "18VC19750913"
# expocode = "320620181021"

In [25]:
# We'll call this version 4 of the direct FORTRAN translation
def _toms510_state_machine_v4(Y, X, tol):
    """
    Pure Python port of the TOMS 510 (STL2) piecewise approximation.
    Implements 1D floating cones, pivot shifting, and backward 
    knot adjustments for slope continuity.
    """
    N = len(Y)
    if N <= 2:
        return [X[i] for i in range(N)]
        
    U = [0.0] * (N + 5)
    V = [0.0] * (N + 5)
    W = [0.0] * (N + 5)
    
    U[0] = float(X[0])
    J = 1
    INIT = 0
    INDC = 0
    KEEP = 0
    I = 0
    SLOPE = 0.0
    KP = 0
    
    E = float(tol)
    SGN = 1.0
    
    XEYE, YEYE, YINIT = 0.0, 0.0, 0.0
    SMIN, SMAX = 0.0, 0.0
    IPIV, IGRAZE = 0, 0
    XINIT = 0.0
    TEMP1, TEMP2 = 0.0, 0.0
    
    state = 30
    
    while True:
        if state == 20:
            J += 1
            INIT = I
            INDC = 0
            KEEP = I
            XEYE = U[J-1]
            YEYE = V[J-1]
            TEMP1 = E
            YINIT = YEYE - 2.0 * E
            state = 40
            
        elif state == 30:
            XEYE = float(X[I])
            YEYE = Y[I] + E
            YINIT = Y[I] - E
            if I == 0:
                state = 40
                continue
            
            SMIN = (Y[I+1] - YEYE - E) / (X[I+1] - XEYE)
            SMAX = (YEYE - Y[I-1] + E) / (XEYE - X[I-1])
            
            if KEEP == I - 1:
                state = 50
                continue
                
            IT = I - 2
            XINIT = XEYE
            IPIV = I
            IGRAZE = I
            I += 1
            state = 150
            
        elif state == 40:
            if XEYE >= X[I]:
                I += 1
            DX = X[I] - XEYE
            SMAX = (Y[I] + E - YEYE) / DX
            SMIN = (Y[I] - E - YEYE) / DX
            state = 50
            
        elif state == 50:
            XINIT = XEYE
            IPIV = I
            IGRAZE = I
            state = 60
            
        elif state == 60:
            if I == N - 1:
                state = 260
                continue
            I += 1
            state = 70
            
        elif state == 70:
            DX = X[I] - XEYE
            TEMP1 = (Y[I] + E - YEYE) / DX
            TEST = -(TEMP1 - SMAX) if SGN <= 0.0 else (TEMP1 - SMAX)
            
            if TEST < 0.0:
                state = 80
            elif TEST == 0.0:
                state = 90
            else:
                state = 100
                
        elif state == 80:
            TEST = -(TEMP1 - SMIN) if SGN <= 0.0 else (TEMP1 - SMIN)
            if TEST < 0.0:
                state = 210
                continue
            SMAX = TEMP1
            state = 90
            
        elif state == 90:
            IPIV = I
            state = 100
            
        elif state == 100:
            TEMP2 = (Y[I] - E - YEYE) / DX
            TEST = -(TEMP2 - SMAX) if SGN <= 0.0 else (TEMP2 - SMAX)
            
            if TEST < 0.0:
                state = 110
            elif TEST == 0.0:
                state = 120
            else:
                state = 140
                
        elif state == 110:
            TEST = -(SMIN - TEMP2) if SGN <= 0.0 else (SMIN - TEMP2)
            if TEST < 0.0:
                state = 120
            elif TEST == 0.0:
                state = 130
            else:
                state = 60
                
        elif state == 120:
            SMIN = TEMP2
            state = 130
            
        elif state == 130:
            IGRAZE = I
            state = 60
            
        elif state == 140:
            if XEYE == X[IPIV]:
                state = 220
                continue
            INDC = 1
            SVX, SVY = XEYE, YEYE
            SVMN, SVMX = SMIN, SMAX
            XEYE = float(X[IPIV])
            YEYE = Y[IPIV] + E  # Corrected sign double-flip
            SMIN = SMAX
            SMAX = (YINIT - YEYE) / (XINIT - XEYE)
            if KEEP >= IPIV:
                state = 170
                continue
            IT = IPIV - 1
            state = 150
            
        elif state == 150:
            TEMP2_val = YEYE + E  # Corrected sign double-flip
            for L in range(KEEP, IT + 1):
                TEMP1_val = (Y[L] - TEMP2_val) / (X[L] - XEYE)
                TEST = -(TEMP1_val - SMAX) if SGN <= 0.0 else (TEMP1_val - SMAX)
                if TEST < 0.0:
                    SMAX = TEMP1_val
            state = 170
            
        elif state == 170:
            if IPIV >= I - 1:
                state = 70
                continue
            IT = I - 2
            TEMP2_val = YEYE - E  # Corrected sign double-flip
            for L in range(IPIV, IT + 1):
                DX_val = X[L+1] - XEYE
                TEMP1_val = (Y[L+1] - TEMP2_val) / DX_val
                TEST = -(TEMP1_val - SMAX) if SGN <= 0.0 else (TEMP1_val - SMAX)
                if TEST < 0.0:
                    SMAX = TEMP1_val
                    IPIV = L + 1
            state = 70
            
        elif state == 210:
            TEMP2 = SMIN
            if I == N - 1:
                state = 240
                continue
            KEEP = IGRAZE
            state = 250
            
        elif state == 220:
            TEMP2 = SMAX
            if I == N - 1:
                state = 230
                continue
            SGN = -SGN
            E = -E
            KEEP = IPIV
            state = 250
            
        elif state == 230:
            if INDC == 0 or XEYE != X[N-2]:
                state = 240
                continue
            XEYE, YEYE = SVX, SVY
            SMIN, SMAX = SVMN, SVMX
            state = 240
            
        elif state == 240:
            U[J] = float(X[N-2])
            YINIT = Y[N-2]
            state = 270
            
        elif state == 250:
            TEMP1 = (Y[I] - Y[I-1]) / (X[I] - X[I-1])
            U[J] = (Y[I] + E - YEYE - TEMP1 * X[I] + TEMP2 * XEYE) / (TEMP2 - TEMP1)
            state = 310
            
        elif state == 260:
            U[J] = float(X[N-1])
            YINIT = Y[N-1]
            state = 270
            
        elif state == 270:
            state = 290
            
        elif state == 290:
            TEMP2 = SMAX
            TEMP1 = YEYE + SMAX * (U[J] - XEYE)
            TEST = -(YINIT - TEMP1) if SGN < 0.0 else (YINIT - TEMP1)
            if TEST > 0.0:
                state = 310
                continue
                
            TEMP2 = SMIN
            TEMP1 = YEYE + SMIN * (U[J] - XEYE)
            TEST = -(YINIT - TEMP1) if SGN < 0.0 else (YINIT - TEMP1)
            if TEST < 0.0:
                state = 310
                continue
                
            TEMP2 = (YINIT - YEYE) / (U[J] - XEYE)
            V[J] = YINIT
            state = 320
            
        elif state == 310:
            V[J] = YEYE + TEMP2 * (U[J] - XEYE)
            state = 320
            
        elif state == 320:
            if XEYE != XINIT:
                state = 330
            else:
                state = 360  # Control IP/ITCH logic for ITCH=2
                
        elif state == 330:
            if J <= 1:
                state = 370
                continue
            if SLOPE == TEMP2:
                state = 360
                continue
                
            YINIT_val = W[J-2]
            TEMP1 = (XEYE * TEMP2 - U[J-2] * SLOPE + YINIT_val - YEYE) / (TEMP2 - SLOPE)
            
            if TEMP1 > XINIT:
                state = 360
                continue
                
            TEST = abs(E)
            IDIOT = INIT - KP
            jump_to_360 = False
            for L_idx in range(1, IDIOT + 1):
                IT = INIT - L_idx
                if TEMP1 >= X[IT]:
                    break
                DX_val = Y[IT] - YEYE - TEMP2 * (X[IT] - XEYE)
                if abs(DX_val) > TEST:
                    jump_to_360 = True
                    break
                    
            if jump_to_360:
                state = 360
            else:
                state = 350
                
        elif state == 350:
            U[J-1] = TEMP1
            V[J-1] = YEYE + TEMP2 * (U[J-1] - XEYE)
            W[J-1] = V[J-1]
            state = 390
            
        elif state == 360:
            W[J-1] = YEYE + TEMP2 * (U[J-1] - XEYE)
            state = 390
            
        elif state == 370:
            state = 360
            
        elif state == 380:
            V[0] = YEYE + TEMP2 * (U[0] - XEYE)
            state = 390
            
        elif state == 390:
            SLOPE = TEMP2
            KP = KEEP
            if I < N - 1:
                state = 20
            else:
                if X[N-1] == U[J]:
                    state = 400
                else:
                    W[J] = V[J]
                    J += 1
                    U[J] = float(X[N-1])
                    V[J] = float(Y[N-1])
                    state = 400
                    
        elif state == 400:
            if J >= 1:
                V[0] = W[0]
            K = J
            break
            
    # Apply Ferret's final DO 500 truncation exactly as in the Fortran
    valid_U = [int(U[i]) for i in range(K + 1)]
    return sorted(list(set(valid_U)))


def get_valid_series(df, col_name):
    """Filters NaNs and bridges missing data with original integer row indices."""
    valid = df[col_name].notna()
    X = np.where(valid)[0].tolist()
    Y = df[col_name][valid].tolist()
    return X, Y


def decimate_socat_pure_python_v4(df, tol_lon=1.0, tol_lat=1.0, tol_fco2=4.0):
    """
    Pure Python drop-in replacement for the PMEL Ferret piecewise3.so library.
    """
    if len(df) < 3:
        return df
        
    X_lon, Y_lon = get_valid_series(df, 'longitude')
    X_lat, Y_lat = get_valid_series(df, 'latitude')
    X_fco2, Y_fco2 = get_valid_series(df, 'fCO2_recommended')
    
    idx_lon = _toms510_state_machine_v4(Y_lon, X_lon, tol_lon) if Y_lon else []
    idx_lat = _toms510_state_machine_v4(Y_lat, X_lat, tol_lat) if Y_lat else []
    idx_fco2 = _toms510_state_machine_v4(Y_fco2, X_fco2, tol_fco2) if Y_fco2 else []
    
    # PMEL Merge (Set Union)
    merged_indices = sorted(list(set(idx_lon) | set(idx_lat) | set(idx_fco2)))
    
    max_idx = len(df) - 1
    merged_indices = [i for i in merged_indices if 0 <= i <= max_idx]
    
    if merged_indices and merged_indices[-1] != max_idx:
        merged_indices.append(max_idx)
        
    return df.iloc[merged_indices].copy()

In [26]:
def _toms510_state_machine(Y, X, tol):
    """
    Pure Python port of the TOMS 510 (STL2) piecewise linear approximation.
    Maintains continuous float precision across segment knots and exactly 
    replicates Fortran's final-stage integer truncation.
    """
    N = len(Y)
    if N <= 2:
        return [X[i] for i in range(N)]
        
    U = [float(X[0])] # Maintain as float!
    E = float(tol)
    
    SGN = 1.0
    KEEP = 0
    I = 0
    
    XEYE, YEYE, YINIT = 0.0, 0.0, 0.0
    SMIN, SMAX = 0.0, 0.0
    IPIV, IGRAZE = 0, 0
    XINIT = 0.0
    TEMP1, TEMP2 = 0.0, 0.0
    
    # Store exact intersection coordinates for segment handoffs
    prev_xeye, prev_yeye = float(X[0]), float(Y[0]) 
    
    state = 30
    
    while True:
        if state == 20:
            # Anchor to the precise continuous intersection knot
            XEYE = prev_xeye
            YEYE = prev_yeye
            YINIT = YEYE - 2.0 * E
            KEEP = I
            state = 40
            
        elif state == 30:
            XEYE = float(X[I])
            YEYE = Y[I] + E
            YINIT = Y[I] - E
            if I == 0:
                state = 40
                continue
            
            SMIN = (Y[I+1] - YEYE - E) / (X[I+1] - XEYE)
            SMAX = (YEYE - Y[I-1] + E) / (XEYE - X[I-1])
            
            if KEEP == I - 1:
                state = 50
                continue
                
            IT = I - 2
            XINIT = XEYE
            IPIV = I
            IGRAZE = I
            I += 1
            state = 150
            
        elif state == 40:
            if XEYE >= X[I]:
                I += 1
            DX = X[I] - XEYE
            SMAX = (Y[I] + E - YEYE) / DX
            SMIN = (Y[I] - E - YEYE) / DX
            state = 50
            
        elif state == 50:
            XINIT = XEYE
            IPIV = I
            IGRAZE = I
            state = 60
            
        elif state == 60:
            if I == N - 1:
                state = 260
                continue
            I += 1
            state = 70
            
        elif state == 70:
            DX = X[I] - XEYE
            TEMP1 = (Y[I] + E - YEYE) / DX
            TEST = -(TEMP1 - SMAX) if SGN <= 0.0 else (TEMP1 - SMAX)
            
            if TEST < 0.0:
                state = 80
            elif TEST == 0.0:
                state = 90
            else:
                state = 100
                
        elif state == 80:
            TEST = -(TEMP1 - SMIN) if SGN <= 0.0 else (TEMP1 - SMIN)
            if TEST < 0.0:
                state = 210
                continue
            SMAX = TEMP1
            state = 90
            
        elif state == 90:
            IPIV = I
            state = 100
            
        elif state == 100:
            TEMP2 = (Y[I] - E - YEYE) / DX
            TEST = -(TEMP2 - SMAX) if SGN <= 0.0 else (TEMP2 - SMAX)
            
            if TEST < 0.0:
                state = 110
            elif TEST == 0.0:
                state = 120
            else:
                state = 140
                
        elif state == 110:
            TEST = -(SMIN - TEMP2) if SGN <= 0.0 else (SMIN - TEMP2)
            if TEST < 0.0:
                state = 120
            elif TEST == 0.0:
                state = 130
            else:
                state = 60
                
        elif state == 120:
            SMIN = TEMP2
            state = 130
            
        elif state == 130:
            IGRAZE = I
            state = 60
            
        elif state == 140:
            if XEYE == X[IPIV]:
                state = 220
                continue
            # Pivot Shifting
            SVX, SVY = XEYE, YEYE
            SVMN, SVMX = SMIN, SMAX
            XEYE = float(X[IPIV])
            YEYE = Y[IPIV] + SGN * E
            SMIN = SMAX
            SMAX = (YINIT - YEYE) / (XINIT - XEYE)
            if KEEP >= IPIV:
                state = 170
                continue
            IT = IPIV - 1
            state = 150
            
        elif state == 150:
            TEMP2_val = YEYE + SGN * E
            for L in range(KEEP, IT + 1):
                TEMP1_val = (Y[L] - TEMP2_val) / (X[L] - XEYE)
                TEST = -(TEMP1_val - SMAX) if SGN <= 0.0 else (TEMP1_val - SMAX)
                if TEST < 0.0:
                    SMAX = TEMP1_val
            state = 170
            
        elif state == 170:
            if IPIV >= I - 1:
                state = 70
                continue
            IT = I - 2
            TEMP2_val = YEYE - SGN * E
            for L in range(IPIV, IT + 1):
                DX_val = X[L+1] - XEYE
                TEMP1_val = (Y[L+1] - TEMP2_val) / DX_val
                TEST = -(TEMP1_val - SMAX) if SGN <= 0.0 else (TEMP1_val - SMAX)
                if TEST < 0.0:
                    SMAX = TEMP1_val
                    IPIV = L + 1
            state = 70
            
        elif state == 210:
            TEMP2 = SMIN
            if I == N - 1:
                state = 240
                continue
            KEEP = IGRAZE
            state = 250
            
        elif state == 220:
            TEMP2 = SMAX
            if I == N - 1:
                state = 230
                continue
            SGN = -SGN
            E = -E
            KEEP = IPIV
            state = 250
            
        elif state == 230:
            if XEYE != X[N-2]:
                state = 240
                continue
            XEYE, YEYE = SVX, SVY
            SMIN, SMAX = SVMN, SVMX
            state = 240
            
        elif state == 240:
            U.append(float(X[N-2]))
            state = 270
            
        elif state == 250:
            # 1. Calculate intersection slopes
            TEMP1 = (Y[I] - Y[I-1]) / (X[I] - X[I-1])
            
            # 2. Calculate continuous float knot exactly matching Fortran math
            knot_x = (Y[I] + E - YEYE - TEMP1 * X[I] + TEMP2 * XEYE) / (TEMP2 - TEMP1)
            knot_y = YEYE + TEMP2 * (knot_x - XEYE)
            
            # 3. Save as float!
            U.append(knot_x)
            
            # 4. Pass the exact continuous coordinates to the next segment
            prev_xeye = knot_x
            prev_yeye = knot_y
            
            state = 310
            
        elif state == 260:
            U.append(float(X[N-1]))
            state = 270
            
        elif state == 270:
            break
            
        elif state == 310:
            if I < N - 1:
                state = 20
            else:
                break
                
    if U[-1] != float(X[N-1]):
        U.append(float(X[N-1]))
        
    # Apply Ferret's final DO 500 truncation exactly as in the Fortran
    final_U = [int(u_val) for u_val in U]
    return sorted(list(set(final_U)))

def get_valid_series(df, col_name):
    """Filters NaNs and returns valid Y data with their original integer row indices (X)."""
    valid = df[col_name].notna()
    X = np.where(valid)[0].tolist()
    Y = df[col_name][valid].tolist()
    return X, Y

def decimate_socat_pure_python(df, tol_lon=1.0, tol_lat=1.0, tol_fco2=4.0):
    if len(df) < 3:
        return df
        
    X_lon, Y_lon = get_valid_series(df, 'longitude')
    X_lat, Y_lat = get_valid_series(df, 'latitude')
    X_fco2, Y_fco2 = get_valid_series(df, 'fCO2_recommended')
    
    idx_lon = _toms510_state_machine(Y_lon, X_lon, tol_lon) if Y_lon else []
    idx_lat = _toms510_state_machine(Y_lat, X_lat, tol_lat) if Y_lat else []
    idx_fco2 = _toms510_state_machine(Y_fco2, X_fco2, tol_fco2) if Y_fco2 else []
    
    merged_indices = sorted(list(set(idx_lon) | set(idx_lat) | set(idx_fco2)))
    
    max_idx = len(df) - 1
    merged_indices = [i for i in merged_indices if 0 <= i <= max_idx]
    
    if merged_indices and merged_indices[-1] != max_idx:
        merged_indices.append(max_idx)
        
    return df.iloc[merged_indices].copy()

In [27]:
def stl2_1d_floating(data_array, tolerance):
    """
    1D Floating Cone Algorithm (TOMS 510).
    Allows the line to be mathematically unanchored (floating) within the +/- tolerance 
    at ALL points, allowing segments to stretch to their absolute mathematical limits.
    """
    N = len(data_array)
    if N < 3:
        return list(range(N))
        
    indices = [0]
    start_idx = 0
    
    while start_idx < N - 1:
        # Initialize the valid slope boundaries for this segment
        m_lower = -float('inf')
        m_upper = float('inf')
        valid_end = start_idx
        
        for curr_idx in range(start_idx + 1, N):
            U_curr = data_array[curr_idx] + tolerance
            L_curr = data_array[curr_idx] - tolerance
            
            new_m_lower = m_lower
            new_m_upper = m_upper
            
            # Check the new point against all previous points in the current segment
            for p in range(start_idx, curr_idx):
                U_p = data_array[p] + tolerance
                L_p = data_array[p] - tolerance
                dx = curr_idx - p
                
                # Minimum slope required to not fall below the current point's lower limit
                slope_lower = (L_curr - U_p) / dx
                if slope_lower > new_m_lower:
                    new_m_lower = slope_lower
                    
                # Maximum slope allowed to not overshoot the current point's upper limit
                slope_upper = (U_curr - L_p) / dx
                if slope_upper < new_m_upper:
                    new_m_upper = slope_upper
            
            # If the constraints cross, the cone snaps shut. No line can exist here!
            if new_m_lower > new_m_upper:
                break
                
            # If valid, update the running bounds and extend the segment
            m_lower = new_m_lower
            m_upper = new_m_upper
            valid_end = curr_idx
            
        if valid_end == start_idx:
            # Failsafe if it couldn't even stretch 1 point
            start_idx += 1
            indices.append(start_idx)
        else:
            indices.append(valid_end)
            start_idx = valid_end
            
    # Always include the last index
    if indices[-1] != N - 1:
        indices.append(N - 1)
        
    return indices

def ferret_piecewise3_exact(df, lon_col='longitude', lat_col='latitude', fco2_col='fCO2_recommended', 
                            tol_lon=1.0, tol_lat=1.0, tol_fco2=4.0):
    """
    Exact structural replica of the PMEL piecewise3.F Fortran logic.
    Runs 1D STL2 independently and merges the resulting indices.
    """
    if len(df) < 3:
        return df
        
    # 1. Extract the 1D arrays as lists (ignoring timestamps/physical distance entirely)
    lons = df[lon_col].tolist()
    lats = df[lat_col].tolist()
    fco2s = df[fco2_col].tolist()
    
    # 2. Run the unanchored 1D algorithm independently on each variable
    idx_lon = stl2_1d_floating(lons, tol_lon)
    idx_lat = stl2_1d_floating(lats, tol_lat)
    idx_fco2 = stl2_1d_floating(fco2s, tol_fco2)
    
    # 3. The Ferret 'Merge' - Set Union of all broken indices
    merged_indices = sorted(list(set(idx_lon) | set(idx_lat) | set(idx_fco2)))
    
    # 4. Return the DataFrame sliced at those exact indices
    return df.iloc[merged_indices].copy()

In [28]:
def stl2_1d(data_array, tolerance):
    """
    1D Wilson Cone Algorithm (TOMS 510).
    Evaluates a single 1D array against its array index.
    """
    if len(data_array) < 3:
        return list(range(len(data_array)))
        
    indices = [0]
    start_idx = 0
    N = len(data_array)
    
    while start_idx < N - 1:
        start_val = data_array[start_idx]
        min_slope = -float('inf')
        max_slope = float('inf')
        valid_end = start_idx
        
        for curr_idx in range(start_idx + 1, N):
            dt = curr_idx - start_idx
            val = data_array[curr_idx]
            
            # Calculate the slope limits for this step
            lower_slope = (val - tolerance - start_val) / dt
            upper_slope = (val + tolerance - start_val) / dt
            
            # Constrain the floating cone
            min_slope = max(min_slope, lower_slope)
            max_slope = min(max_slope, upper_slope)
            
            # If the cone closes, the line breaks
            if min_slope > max_slope:
                break
                
            valid_end = curr_idx
            
        if valid_end == start_idx:
            start_idx += 1
            indices.append(start_idx)
        else:
            indices.append(valid_end)
            start_idx = valid_end
            
    # Always include the last index
    if indices[-1] != N - 1:
        indices.append(N - 1)
        
    return indices


def ferret_piecewise3_exact(path, tol_lon=1.0, tol_lat=1.0, tol_fco2=4.0):
    """
    Exact structural replica of the PMEL piecewise3.F Fortran logic.
    Runs 1D STL2 independently and merges the resulting indices.
    """
    if len(path) < 3:
        return path
        
    # 1. Extract the 1D arrays
    lons = [pt.lon for pt in path]
    lats = [pt.lat for pt in path]
    fco2s = [pt.fCO2_recommended for pt in path]
    
    # 2. Run the 1D algorithm independently on each variable
    idx_lon = stl2_1d(lons, tol_lon)
    idx_lat = stl2_1d(lats, tol_lat)
    idx_fco2 = stl2_1d(fco2s, tol_fco2)
    
    # 3. The Ferret 'Merge' - Set Union of all broken indices
    merged_indices = sorted(list(set(idx_lon) | set(idx_lat) | set(idx_fco2)))
    
    # 4. Return the data at those exact indices
    return [path[i] for i in merged_indices]

In [29]:
def ferret_piecewise3_rmse(path, tol_lon=1.0, tol_lat=1.0, tol_fco2=4.0):
    """
    Exact replication of the PMEL Ferret piecewise3.so (control=2) algorithm.
    Uses a Greedy L2-Norm (Root Mean Square Error) piecewise interpolation.
    This allows the segment to absorb massive local sensor spikes as long as 
    the overall mathematical average of the segment remains within the tolerances.
    """
    if len(path) < 3:
        return path
        
    simplified = [path[0]]
    start_idx = 0
    N = len(path)
    
    while start_idx < N - 1:
        valid_end_idx = start_idx + 1
        
        # Greedily stretch the segment as far forward as possible
        for curr_idx in range(start_idx + 2, N):
            is_valid = True
            
            p1 = path[start_idx]
            p2 = path[curr_idx]
            dt_total = curr_idx - start_idx
            
            sum_sq_error = 0.0
            
            # Evaluate the L2 (RMSE) error of all intermediate points
            for k in range(start_idx + 1, curr_idx):
                pt = path[k]
                fraction = (k - start_idx) / dt_total
                
                # 1. Interpolated expected values along the straight line
                exp_lon = p1.lon + fraction * (p2.lon - p1.lon)
                exp_lat = p1.lat + fraction * (p2.lat - p1.lat)
                exp_fco2 = p1.fCO2_recommended + fraction * (p2.fCO2_recommended - p1.fCO2_recommended)
                
                # 2. Normalize the errors against your Ferret tolerances
                err_lon = (pt.lon - exp_lon) / tol_lon
                err_lat = (pt.lat - exp_lat) / tol_lat
                err_fco2 = (pt.fCO2_recommended - exp_fco2) / tol_fco2
                
                # 3. Accumulate squared Euclidean distance in 3D scaled space
                sum_sq_error += (err_lon**2 + err_lat**2 + err_fco2**2)
            
            # 4. Calculate Root Mean Square Error (RMSE)
            num_intermediate_points = curr_idx - start_idx - 1
            rmse = math.sqrt(sum_sq_error / num_intermediate_points)
            
            # The Ferret control=2 condition: Does the AVERAGE segment error exceed 1.0?
            if rmse >= 1.0:
                is_valid = False
                break
                
            if is_valid:
                valid_end_idx = curr_idx
            else:
                break # Segment average broke tolerance; drop waypoint at previous index
                
        simplified.append(path[valid_end_idx])
        start_idx = valid_end_idx
        
    # Always ensure the last point of the dataset is included
    if simplified[-1] != path[-1]:
        simplified.append(path[-1])
        
    return simplified

In [30]:
# Visual/Trajectory Smoothing (Ferret replication): If your goal is purely to mimic the legacy PMEL dataset sizes 
# (148 points) and you are okay with the algorithm internally "smoothing over" 10+ µatm spikes, use the ferret_piecewise3_replicator above.
def ferret_piecewise3_replicator(path, tol_lon=1.0, tol_lat=1.0, tol_fco2=4.0):
    """
    Replicates the aggressive data reduction of the Ferret piecewise3.so (control=2) 
    library. It uses a relaxed, normalized 3D error heuristic that allows localized 
    sensor spikes to be smoothed over long physical trajectories.
    """
    if len(path) < 3:
        return path
        
    simplified = [path[0]]
    start_idx = 0
    N = len(path)
    
    while start_idx < N - 1:
        valid_end_idx = start_idx + 1
        
        # Greedy stretch: push the segment as far forward as possible
        for curr_idx in range(start_idx + 2, N):
            is_valid = True
            
            p1 = path[start_idx]
            p2 = path[curr_idx]
            
            # The independent variable (time/index steps)
            dt_total = curr_idx - start_idx
            
            # Evaluate the overall "drift" of the segment
            for k in range(start_idx + 1, curr_idx):
                pt = path[k]
                fraction = (k - start_idx) / dt_total
                
                # 1. Calculate the interpolated expected values
                exp_lon = p1.lon + fraction * (p2.lon - p1.lon)
                exp_lat = p1.lat + fraction * (p2.lat - p1.lat)
                exp_fco2 = p1.fCO2_recommended + fraction * (p2.fCO2_recommended - p1.fCO2_recommended)
                
                # 2. Normalize the errors against your Ferret tolerances
                err_lon = abs(pt.lon - exp_lon) / tol_lon
                err_lat = abs(pt.lat - exp_lat) / tol_lat
                err_fco2 = abs(pt.fCO2_recommended - exp_fco2) / tol_fco2
                
                # 3. The Ferret "Cheat": Normalized Euclidean Magnitude
                # Instead of strict bounding (where any single error > 1.0 breaks the line),
                # we calculate the combined 3D magnitude. This allows a spike in fCO2 
                # to be "forgiven" if the Lat/Lon trajectory is perfectly straight.
                magnitude_error = math.sqrt(err_lon**2 + err_lat**2 + err_fco2**2)
                
                # If the overall magnitude pulls too hard away from the trajectory, break the line
                # Note: Adjusting this threshold (e.g., to 1.5 or 2.0) dictates how "forgiving" 
                # the algorithm is to spikes, directly mirroring the Ferret C-library internals.
                if magnitude_error > 1.5:  
                    is_valid = False
                    break
            
            if is_valid:
                valid_end_idx = curr_idx
            else:
                break # The segment finally snapped, drop our waypoint and restart
                
        simplified.append(path[valid_end_idx])
        start_idx = valid_end_idx
        
    return simplified

In [31]:
# Strict Data Integrity: If it is critical that no data point between your waypoints deviates by more than 4 µatm, 
# you should stick to the wilson_distance_based script from the previous prompt. It gave 7 points because it correctly caught 
# the 342.57 drop at 00:44 and forced a waypoint to protect your data (using expocode="11BE19950303")
def wilson_distance_based(path, tol_lon=1.0, tol_lat=1.0, tol_fco2=4.0):
    """
    Implements the Wilson Cone Algorithm using cumulative physical distance 
    as the independent parameter, matching the likely behavior of control=2.
    """
    if len(path) < 3:
        return path
        
    simplified = [path[0]]
    start_idx = 0
    
    while start_idx < len(path) - 1:
        start_pt = path[start_idx]
        
        min_slope_lon, max_slope_lon = -float('inf'), float('inf')
        min_slope_lat, max_slope_lat = -float('inf'), float('inf')
        min_slope_fco2, max_slope_fco2 = -float('inf'), float('inf')
        
        valid_end_idx = start_idx
        cumulative_dist = 0.0  # Tracks distance from the start point of this segment
        
        for curr_idx in range(start_idx + 1, len(path)):
            pt = path[curr_idx]
            prev_pt = path[curr_idx - 1]
            
            # 1. Update the independent variable (dt) with physical distance in meters
            cumulative_dist += prev_pt.distanceTo(pt)
            dt = cumulative_dist
            
            # Safeguard: If the ship is perfectly stationary, distance is 0.
            # We skip slope calculation to avoid division by zero, but still check if fCO2 spiked.
            if dt == 0.0:
                if abs(pt.fCO2_recommended - start_pt.fCO2_recommended) > tol_fco2:
                    break
                valid_end_idx = curr_idx
                continue
            
            # 2. fCO2 Cone Limits (Slope is now: µatm change per meter traveled)
            fco2_lower = (pt.fCO2_recommended - tol_fco2 - start_pt.fCO2_recommended) / dt
            fco2_upper = (pt.fCO2_recommended + tol_fco2 - start_pt.fCO2_recommended) / dt
            min_slope_fco2 = max(min_slope_fco2, fco2_lower)
            max_slope_fco2 = min(max_slope_fco2, fco2_upper)
            
            # 3. Latitude Cone Limits (Slope is now: degree change per meter traveled)
            lat_lower = (pt.lat - tol_lat - start_pt.lat) / dt
            lat_upper = (pt.lat + tol_lat - start_pt.lat) / dt
            min_slope_lat = max(min_slope_lat, lat_lower)
            max_slope_lat = min(max_slope_lat, lat_upper)
            
            # 4. Longitude Cone Limits
            lon_delta = (pt.lon - start_pt.lon + 180) % 360 - 180
            lon_lower = (lon_delta - tol_lon) / dt
            lon_upper = (lon_delta + tol_lon) / dt
            min_slope_lon = max(min_slope_lon, lon_lower)
            max_slope_lon = min(max_slope_lon, lon_upper)
            
            # If ANY of the cones close, the line breaks
            if (min_slope_fco2 > max_slope_fco2) or \
               (min_slope_lat > max_slope_lat) or \
               (min_slope_lon > max_slope_lon):
                break 
                
            valid_end_idx = curr_idx
            
        # Append the last successful index and move the start point
        if valid_end_idx == start_idx:
            start_idx += 1
            if start_idx < len(path):
                simplified.append(path[start_idx])
        else:
            simplified.append(path[valid_end_idx])
            start_idx = valid_end_idx
            
    if simplified[-1] != path[-1]:
        simplified.append(path[-1])
        
    return simplified

In [32]:
def true_wilson_piecewise3(path, tol_lon=1.0, tol_lat=1.0, tol_fco2=4.0):
    """
    Implements the true Wilson (1972) 'Fewest Line Segments' Cone Algorithm.
    Calculates an expanding 'cone' of valid slopes to allow segments to stretch 
    as far as mathematically possible without breaking tolerances.
    """
    if len(path) < 3:
        return path
        
    simplified = [path[0]]
    start_idx = 0
    
    while start_idx < len(path) - 1:
        start_pt = path[start_idx]
        
        # Initialize the valid slope ranges (the "cone") for all 3 variables
        # Slopes represent: change in value per step forward
        min_slope_lon, max_slope_lon = -float('inf'), float('inf')
        min_slope_lat, max_slope_lat = -float('inf'), float('inf')
        min_slope_fco2, max_slope_fco2 = -float('inf'), float('inf')
        
        valid_end_idx = start_idx
        
        for curr_idx in range(start_idx + 1, len(path)):
            pt = path[curr_idx]
            
            # The independent variable (x-axis for the slope) is the array index
            dt = curr_idx - start_idx  
            
            # 1. fCO2 Cone Limits
            fco2_lower = (pt.fCO2_recommended - tol_fco2 - start_pt.fCO2_recommended) / dt
            fco2_upper = (pt.fCO2_recommended + tol_fco2 - start_pt.fCO2_recommended) / dt
            min_slope_fco2 = max(min_slope_fco2, fco2_lower)
            max_slope_fco2 = min(max_slope_fco2, fco2_upper)
            
            # 2. Latitude Cone Limits
            lat_lower = (pt.lat - tol_lat - start_pt.lat) / dt
            lat_upper = (pt.lat + tol_lat - start_pt.lat) / dt
            min_slope_lat = max(min_slope_lat, lat_lower)
            max_slope_lat = min(max_slope_lat, lat_upper)
            
            # 3. Longitude Cone Limits (Handles the International Date Line wrap)
            lon_delta = (pt.lon - start_pt.lon + 180) % 360 - 180
            lon_lower = (lon_delta - tol_lon) / dt
            lon_upper = (lon_delta + tol_lon) / dt
            min_slope_lon = max(min_slope_lon, lon_lower)
            max_slope_lon = min(max_slope_lon, lon_upper)
            
            # If ANY of the cones close (min > max), a line can no longer exist!
            if (min_slope_fco2 > max_slope_fco2) or \
               (min_slope_lat > max_slope_lat) or \
               (min_slope_lon > max_slope_lon):
                break 
                
            # If we make it here, the cones are still open. We can stretch further!
            valid_end_idx = curr_idx
            
        # Append the last successful index
        simplified.append(path[valid_end_idx])
        
        # Move our start point forward
        if valid_end_idx == start_idx:
            # Failsafe: if we couldn't even move 1 point (shouldn't happen with valid data)
            start_idx += 1
            if start_idx < len(path):
                simplified.append(path[start_idx])
        else:
            start_idx = valid_end_idx
            
    # Always ensure the last point of the dataset is included
    if simplified[-1] != path[-1]:
        simplified.append(path[-1])
        
    return simplified

In [33]:
def wilson_piecewise3(path, tol_lon=1.0, tol_lat=1.0, tol_fco2=4.0):
    """
    Mimics the PMEL/Ferret piecewise3.so sequential algorithm.
    Interpolates Lon, Lat, and fCO2 linearly based on array index.
    """
    if len(path) < 3:
        return path
        
    simplified = [path[0]]
    start_idx = 0
    
    while start_idx < len(path) - 1:
        valid_end_idx = start_idx + 1
        
        # Try to stretch the segment as far forward as possible
        for curr_idx in range(start_idx + 2, len(path)):
            is_valid = True
            
            start_pt = path[start_idx]
            end_pt = path[curr_idx]
            
            # The total number of "steps" in this stretch
            total_steps = curr_idx - start_idx
            
            # Check every intermediate point
            for k in range(start_idx + 1, curr_idx):
                pt = path[k]
                
                # Calculate what fraction of the way we are from start to end
                fraction = (k - start_idx) / total_steps
                
                # 1. Evaluate fCO2 Interpolation Error
                expected_fco2 = start_pt.fCO2_recommended + fraction * (end_pt.fCO2_recommended - start_pt.fCO2_recommended)
                if abs(pt.fCO2_recommended - expected_fco2) > tol_fco2:
                    is_valid = False
                    break
                    
                # 2. Evaluate Latitude Interpolation Error
                expected_lat = start_pt.lat + fraction * (end_pt.lat - start_pt.lat)
                if abs(pt.lat - expected_lat) > tol_lat:
                    is_valid = False
                    break
                    
                # 3. Evaluate Longitude Interpolation Error
                # (Safely handles wraparound over the International Date Line)
                lon_diff = (end_pt.lon - start_pt.lon + 180) % 360 - 180
                expected_lon = start_pt.lon + fraction * lon_diff
                
                lon_error = abs((pt.lon - expected_lon + 180) % 360 - 180)
                if lon_error > tol_lon:
                    is_valid = False
                    break
            
            # If the stretch to curr_idx was mathematically valid, save the index and try going one further
            if is_valid:
                valid_end_idx = curr_idx
            else:
                # The line broke! Drop our waypoint and start over.
                break 
        
        simplified.append(path[valid_end_idx])
        start_idx = valid_end_idx
        
    return simplified

In [34]:
def simplify_sequential_box_and_fco2(path, lat_tolerance=1.0, lon_tolerance=1.0, fco2_tolerance=4.0):
    """
    Greedily stretches a line segment as far as possible until the interpolated fCO2
    exceeds tolerance, OR any point exits a Lat/Lon bounding box centered on the segment start.
    """
    if len(path) < 3:
        return path
        
    simplified = [path[0]]
    start_idx = 0
    
    while start_idx < len(path) - 1:
        valid_end_idx = start_idx + 1
        
        for curr_idx in range(start_idx + 2, len(path)):
            is_valid = True
            
            start_pt = path[start_idx]
            end_pt = path[curr_idx]
            total_dist = start_pt.distanceTo(end_pt)
            
            for k in range(start_idx + 1, curr_idx):
                pt = path[k]
                
                # 1. Check fCO2 Deviation FIRST
                if total_dist > 0:
                    fraction = start_pt.distanceTo(pt) / total_dist
                else:
                    fraction = 0.0
                    
                expected_fco2 = start_pt.fCO2_recommended + fraction * (end_pt.fCO2_recommended - start_pt.fCO2_recommended)
                
                if abs(pt.fCO2_recommended - expected_fco2) > fco2_tolerance:
                    is_valid = False
                    break 
                    
                # 2. Check Bounding Box Deviation SECOND
                lat_diff = abs(pt.lat - start_pt.lat)
                lon_diff = abs((pt.lon - start_pt.lon + 180) % 360 - 180)
                
                if lat_diff > lat_tolerance or lon_diff > lon_tolerance:
                    is_valid = False
                    break 
            
            if is_valid:
                valid_end_idx = curr_idx
            else:
                break 
        
        simplified.append(path[valid_end_idx])
        start_idx = valid_end_idx
        
    return simplified

In [35]:
def simplify_sequential_fco2(path, dist_tolerance_m=111120.0, fco2_tolerance=4.0):
    """
    Greedily stretches a line segment as far as possible until the interpolated fCO2
    OR the cross-track distance of any intermediate point exceeds the tolerance.
    """
    if len(path) < 3:
        return path
        
    simplified = [path[0]]
    start_idx = 0
    
    while start_idx < len(path) - 1:
        valid_end_idx = start_idx + 1
        
        # Try to stretch the segment as far forward as possible
        for curr_idx in range(start_idx + 2, len(path)):
            is_valid = True
            
            start_pt = path[start_idx]
            end_pt = path[curr_idx]
            total_dist = start_pt.distanceTo(end_pt)
            
            # Check every point between our start and our current stretch target
            for k in range(start_idx + 1, curr_idx):
                pt = path[k]
                
                # 1. Check fCO2 Deviation FIRST (Prioritizing the sensor data)
                if total_dist > 0:
                    fraction = start_pt.distanceTo(pt) / total_dist
                else:
                    fraction = 0.0
                    
                expected_fco2 = start_pt.fCO2_recommended + fraction * (end_pt.fCO2_recommended - start_pt.fCO2_recommended)
                
                if abs(pt.fCO2_recommended - expected_fco2) >= fco2_tolerance:
                    is_valid = False
                    break # The fCO2 tolerance broke, stop checking this stretch
                    
                # 2. Check Spatial Deviation SECOND
                try:
                    cross_track = abs(pt.crossTrackDistanceTo(start_pt, end_pt))
                except ValueError:
                    cross_track = 0.0
                    
                if cross_track > dist_tolerance_m:
                    is_valid = False
                    break # The spatial tolerance broke, stop checking this stretch
            
            # If the stretch to curr_idx was valid, save it and try stretching one more
            if is_valid:
                valid_end_idx = curr_idx
            else:
                # The line broke! We cannot stretch to curr_idx. 
                # We must drop our waypoint at the last known good index.
                break
        
        # Add the furthest successful point to our final track
        simplified.append(path[valid_end_idx])
        # Start the next segment exactly where this one ended
        start_idx = valid_end_idx
        
    return simplified

In [36]:
def rdp_fco2(path, dist_tolerance_m=111120.0, fco2_tolerance=4.0):
    """
    Recursively simplifies a path using a modified Ramer-Douglas-Peucker algorithm.
    Splits the segment if spatial cross-track distance OR interpolated fCO2 deviates past tolerances.
    """
    # Base case: A line segment needs at least 3 points to have a middle point to evaluate
    if len(path) < 3:
        return path

    start_pt = path[0]
    end_pt = path[-1]

    max_violation_ratio = 0.0
    split_index = 0

    # Calculate total distance to use for fCO2 interpolation weighting
    total_dist = start_pt.distanceTo(end_pt)

    # Evaluate every intermediate point
    for i in range(1, len(path) - 1):
        pt = path[i]

        # 1. Evaluate Spatial Deviation (Cross-Track Distance in meters)
        try:
            # abs() used because cross-track distance can be negative (left/right of track)
            cross_track_dist = abs(pt.crossTrackDistanceTo(start_pt, end_pt))
        except ValueError:
            # Failsafe if start and end points are identical
            cross_track_dist = 0.0

        # 2. Evaluate fCO2 Deviation (Linear Interpolation)
        if total_dist > 0:
            # How far along the track is this point?
            fraction_traveled = start_pt.distanceTo(pt) / total_dist
        else:
            fraction_traveled = 0.0
            
        expected_fco2 = start_pt.fCO2_recommended + fraction_traveled * (end_pt.fCO2_recommended - start_pt.fCO2_recommended)
        fco2_diff = abs(pt.fCO2_recommended - expected_fco2)

        # 3. Normalize the deviations to see which point is the "worst offender"
        # A ratio > 1.0 means it broke the tolerance boundary
        dist_ratio = cross_track_dist / dist_tolerance_m
        fco2_ratio = fco2_diff / fco2_tolerance
        
        point_max_ratio = max(dist_ratio, fco2_ratio)

        # Track the point that deviates the most severely
        if point_max_ratio > max_violation_ratio:
            max_violation_ratio = point_max_ratio
            split_index = i

    # 4. If the worst point exceeds tolerances, split the path and recurse
    if max_violation_ratio > 1.0:
        left_split = rdp_fco2(path[:split_index + 1], dist_tolerance_m, fco2_tolerance)
        right_split = rdp_fco2(path[split_index:], dist_tolerance_m, fco2_tolerance)
        
        # Combine the results, stripping the duplicate point at the split index
        return left_split[:-1] + right_split
        
    else:
        # If no points violate the tolerance, return just the start and end points!
        return [start_pt, end_pt]


In [37]:
def simplify_by_latlon_box_and_fCO2(path, lat_tolerance=1.0, lon_tolerance=1.0, fco2_tolerance=4.0):
    """
    Reduces a path by keeping points that exit a defined Lat/Lon box
    relative to the last kept point, OR deviate in fCO2 by > fco2_tolerance.
    """
    if len(path) < 2:
        return path
        
    simplified = [path[0]]
    last_kept = path[0]
    
    for i in range(1, len(path)):
        pt = path[i]
        
        # 1. Calculate the latitude difference (straightforward absolute difference)
        lat_diff = abs(pt.lat - last_kept.lat)
        
        # 2. Calculate longitude difference (safely handles the 180/-180 antimeridian wraparound)
        lon_diff = abs((pt.lon - last_kept.lon + 180) % 360 - 180)
        
        # 3. Calculate the fCO2 difference
        try:
            fco2_diff = abs(pt.fCO2_recommended - last_kept.fCO2_recommended)
        except AttributeError:
            raise ValueError("Data points must have an 'fCO2_recommended' attribute.")
        
        # 4. Apply the bounding box OR fCO2 conditions
        if lat_diff > lat_tolerance or lon_diff > lon_tolerance or fco2_diff > fco2_tolerance:
            simplified.append(pt)
            last_kept = pt
            
    # Always ensure the final point of the original path is included
    if simplified[-1] != path[-1]:
        simplified.append(path[-1])
        
    return simplified

In [38]:
def simplify_by_heading_and_fCO2(path, angle_tolerance_deg=1.0, fco2_tolerance=4.0):
    """
    Reduces a path by keeping points that deviate in heading by > angle_tolerance_deg
    OR deviate in fCO2_recommended by > fco2_tolerance.
    """
    if len(path) < 3:
        return path
        
    simplified = [path[0]]
    last_kept = path[0]
    
    # Establish the initial track bearing (from point 0 to point 1)
    last_bearing = last_kept.initialBearingTo(path[1])
    
    for i in range(1, len(path)):
        pt = path[i]
        
        # 1. Calculate the bearing from the last kept point to the current point
        current_bearing = last_kept.initialBearingTo(pt)
        
        # Calculate shortest angular difference (handles 359 to 1 degree wraparound)
        angle_diff = abs((current_bearing - last_bearing + 180) % 360 - 180)
        
        # 2. Calculate the fCO2 difference
        # Assuming the attribute 'fCO2_recommended' has been added to the points
        try:
            fco2_diff = abs(pt.fCO2_recommended - last_kept.fCO2_recommended)
        except AttributeError:
            raise ValueError("Data points must have an 'fCO2_recommended' attribute.")
        
        # 3. Apply the conditions (Heading > 1 deg OR fCO2 > 4)
        if angle_diff > angle_tolerance_deg or fco2_diff > fco2_tolerance:
            simplified.append(pt)
            last_kept = pt
            last_bearing = current_bearing  # Update the track bearing reference
            
    # Always ensure the final point of the original path is included
    if simplified[-1] != path[-1]:
        simplified.append(path[-1])
        
    return simplified

In [39]:
data_url = f'{full_url}.csv?time,expocode,latitude,longitude,fCO2_recommended&expocode="{expocode}"&orderBy("time")'
data_url

'https://data.pmel.noaa.gov/socat/erddap/tabledap/socat_v2025_fulldata.csv?time,expocode,latitude,longitude,fCO2_recommended&expocode="11BE19950303"&orderBy("time")'

In [40]:
df = pd.read_csv(data_url, skiprows=[1])
df['source'] = 'full'
with open('ori.txt', "w") as fp:
    for it, row in df.iterrows():
        if row['time'] > '1995-03-04T09:00':
            break
        fp.write(f"{row['time']}, {row['latitude']}, {row['longitude']}, {row['fCO2_recommended']}\n")

In [41]:
path = []
for index, row in df.iterrows():
    point = LatLon(row['latitude'], row['longitude'])
    point.fCO2_recommended = row['fCO2_recommended']
    path.append(point)

In [42]:
filtered_df = decimate_socat_pure_python_v4(df)
# coordinates_tuples = [(p.lat, p.lon) for p in simplified]
# df['coords'] = list(zip(df['latitude'], df['longitude']))
# filtered_df = df[df['coords'].isin(coordinates_tuples)]
# filtered_df = filtered_df.drop(columns=['coords'])
filtered_df['source'] = 'Filtered with pure Python'
print(len(filtered_df))
with open('pure_python.csv', "w") as fp:
    fp.write('time,latitude,longitude,fCO2_recommended\n')
    for rit, rrow in filtered_df.iterrows():
        if rrow['time'] > '1995-03-04T09:00':
            break
        fp.write(f"{rrow['time']}, {row['latitude']}, {rrow['longitude']}, {rrow['fCO2_recommended']}\n")

148


In [43]:
dec_url = f'{decimated_url}.csv?time,expocode,latitude,longitude,fCO2_recommended&expocode="{expocode}"&orderBy("time")'
dec = pd.read_csv(dec_url, skiprows=[1])
dec['source'] = 'Decimated with Ferret'
print(len(dec))
with open('ferret.csv', "w") as fp:
    fp.write('time,latitude,longitude,fCO2_recommended\n')
    for rit, rrow in dec.iterrows():
        if rrow['time'] > '1995-03-04T09:00':
            break
        fp.write(f"{rrow['time']}, {row['latitude']}, {rrow['longitude']}, {rrow['fCO2_recommended']}\n")

148


In [44]:
import plotly.express as px
both = pd.concat([dec, filtered_df])
figure = px.line(both, x='time', y='fCO2_recommended', color='source', title=f'Decimation comparison for {expocode}')
figure.update_layout(margin={'t': 80})
figure